In [ ]:
!pip install -q kaggle

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import mixed_precision

In [ ]:
# Clear old TensorFlow graphs
tf.keras.backend.clear_session()

# Enable mixed precision (saves huge GPU memory)
mixed_precision.set_global_policy('mixed_float16')

Mixed Precision Enabled


In [ ]:
print("TensorFlow Version:", tf.__version__)

print("GPU Available:",
      tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import files

files.upload()

Saving kaggle (3).json to kaggle (3).json


{'kaggle (3).json': b'{"username":"riyashakyahaha","key":"49a1c7951ed820cc8f818608a819ef5a"}'}

In [ ]:
!mkdir -p ~/.kaggle

!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d briscdataset/brisc2025

Dataset URL: https://www.kaggle.com/datasets/briscdataset/brisc2025
License(s): Attribution 4.0 International (CC BY 4.0)
100% 250M/250M [00:03<00:00, 86.6MB/s]



In [ ]:
with zipfile.ZipFile(
    "brisc2025.zip",
    'r'
) as zip_ref:

    zip_ref.extractall(
        "/content/brisc2025"
    )

In [ ]:
# Dataset Paths
image_path = "/content/brisc2025/brisc2025/segmentation_task/train/images"
mask_path = "/content/brisc2025/brisc2025/segmentation_task/train/masks"


# MEMORY SAFE CONFIGURATION
IMG_HEIGHT = 128
IMG_WIDTH = 128

IMG_CHANNELS = 3


# VERY IMPORTANT FOR COLAB RAM
BATCH_SIZE = 8

# Better longer training
EPOCHS = 1000


print("Configuration Ready")

Configuration Ready


In [ ]:
image_files = sorted(
    os.listdir(image_path)
)

image_mask_pairs = []

for img_file in image_files:

    base_name = os.path.splitext(
        img_file
    )[0]

    mask_file = base_name + ".png"

    image_full_path = os.path.join(
        image_path,
        img_file
    )

    mask_full_path = os.path.join(
        mask_path,
        mask_file
    )

    if os.path.exists(mask_full_path):

        image_mask_pairs.append(
            (
                image_full_path,
                mask_full_path
            )
        )

print("Total Valid Pairs:",
      len(image_mask_pairs))

Total Valid Pairs: 3933


In [ ]:
train_pairs, val_pairs = train_test_split(

    image_mask_pairs,

    test_size=0.2,

    random_state=42,

    shuffle=True
)

print("Training Samples:",
      len(train_pairs))

print("Validation Samples:",
      len(val_pairs))

Training Samples: 3146
Validation Samples: 787


In [ ]:
def load_data(image_path, mask_path):

    # READ IMAGE
    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    ) # Removed [:, :, 0] as it reduces dimensions causing error.

    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 255.0


    # READ MASK
    mask = tf.io.read_file(mask_path)

    mask = tf.image.decode_png(
        mask,
        channels=1
    )

    # VERY IMPORTANT FIX
    mask = tf.image.resize(

        mask,

        [IMG_HEIGHT, IMG_WIDTH],

        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )

    # Convert mask to binary
    mask = tf.cast(
        mask > 0,
        tf.float32
    )

    return image, mask

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(

    (
        [x[0] for x in train_pairs],

        [x[1] for x in train_pairs]
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(

    (
        [x[0] for x in val_pairs],

        [x[1] for x in val_pairs]
    )
)


# LOAD DATA
train_dataset = train_dataset.map(
    load_data,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_data,
    num_parallel_calls=tf.data.AUTOTUNE
)


# OPTIMISED PIPELINE
train_dataset = (

    train_dataset

    .shuffle(len(train_pairs))

    .batch(BATCH_SIZE)

    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (

    val_dataset

    .batch(BATCH_SIZE)

    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset Pipeline Ready")

ValueError: in user code:

    File "/tmp/ipykernel_6609/1918878049.py", line 11, in load_data  *
        image = tf.image.resize(

    ValueError: 'images' must have either 3 or 4 dimensions.


In [ ]:
for images, masks in train_dataset.take(1):

    plt.figure(figsize=(12,6))

    for i in range(BATCH_SIZE):

        # MRI IMAGE
        plt.subplot(2,3,i+1)

        plt.imshow(images[i])

        plt.title("MRI Image")

        plt.axis("off")


        # MASK
        plt.subplot(2,3,i+4)

        plt.imshow(

            tf.squeeze(masks[i]),

            cmap='gray'
        )

        plt.title("Tumor Mask")

        plt.axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
def conv_block(inputs, filters):

    x = layers.Conv2D(

        filters,

        3,

        padding="same"
    )(inputs)

    x = layers.BatchNormalization()(x)

    x = layers.Activation("relu")(x)


    x = layers.Conv2D(

        filters,
        3,
        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    return x

In [ ]:
def encoder_block(inputs, filters):

    x = conv_block(
        inputs,
        filters
    )

    p = layers.MaxPooling2D(
        (2,2)
    )(x)

    return x, p

In [ ]:
def decoder_block(

    inputs,

    skip_features,

    filters
):

    x = layers.Conv2DTranspose(

        filters,

        2,

        strides=2,

        padding="same"

    )(inputs)

    x = layers.Concatenate()(
        [x, skip_features]
    )

    x = conv_block(
        x,
        filters
    )

    return x

In [ ]:
def build_unet(

    input_shape=(
        IMG_HEIGHT,
        IMG_WIDTH,
        IMG_CHANNELS
    )
):

    inputs = layers.Input(input_shape)


    # ENCODER
    s1, p1 = encoder_block(inputs, 16)

    s2, p2 = encoder_block(p1, 32)

    s3, p3 = encoder_block(p2, 64)

    s4, p4 = encoder_block(p3, 128)


    # BOTTLENECK
    b1 = conv_block(p4, 256)


    # DECODER
    d1 = decoder_block(b1, s4, 128)

    d2 = decoder_block(d1, s3, 64)

    d3 = decoder_block(d2, s2, 32)

    d4 = decoder_block(d3, s1, 16)


    # OUTPUT
    outputs = layers.Conv2D(

        1,

        1,

        padding="same",

        activation="sigmoid",

        dtype='float32'

    )(d4)


    model = models.Model(
        inputs,
        outputs
    )

    return model

In [ ]:
def dice_coefficient(

    y_true,

    y_pred,

    smooth=1e-6
):

    y_true_f = tf.keras.backend.flatten(
        y_true
    )

    y_pred_f = tf.keras.backend.flatten(
        y_pred
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    return (

        2.0 * intersection + smooth

    ) / (

        tf.reduce_sum(y_true_f)

        + tf.reduce_sum(y_pred_f)

        + smooth
    )

In [ ]:
def dice_loss(y_true, y_pred):

    return 1 - dice_coefficient(
        y_true,
        y_pred
    )

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()

def combined_loss(

    y_true,

    y_pred
):

    return (

        bce(y_true, y_pred)

        + dice_loss(y_true, y_pred)
    )

In [ ]:
unet_model = build_unet()

unet_model.summary()

In [ ]:
unet_model.compile(

    optimizer=tf.keras.optimizers.Adam(

        learning_rate=1e-4
    ),

    loss=combined_loss,

    metrics=[

        dice_coefficient,

        tf.keras.metrics.BinaryIoU(

            target_class_ids=[1],

            threshold=0.5
        ),

        tf.keras.metrics.Precision(),

        tf.keras.metrics.Recall()
    ]
)

In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(

        monitor='val_loss',

        patience=8,

        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(

        monitor='val_loss',

        factor=0.5,

        patience=3,

        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(

        "best_unet_model.h5",

        monitor='val_loss',

        save_best_only=True
    )
]

In [ ]:
history = unet_model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(

    history.history['dice_coefficient']
)

plt.plot(

    history.history['val_dice_coefficient']
)

plt.title("Dice Coefficient Curve")

plt.xlabel("Epoch")

plt.ylabel("Dice Score")

plt.legend([

    "Train",

    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(

    history.history['binary_io_u']
)

plt.plot(

    history.history['val_binary_io_u']
)

plt.title("IoU Curve")

plt.xlabel("Epoch")

plt.ylabel("IoU Score")

plt.legend([

    "Train",

    "Validation"
])

plt.show()

In [ ]:
for images, masks in val_dataset.take(1):

    predictions = unet_model.predict(images)

    plt.figure(figsize=(15,10))

    for i in range(BATCH_SIZE):

        # INPUT IMAGE
        plt.subplot(BATCH_SIZE,3,i*3+1)

        plt.imshow(images[i])

        plt.title("Input MRI")

        plt.axis("off")


        # GROUND TRUTH
        plt.subplot(BATCH_SIZE,3,i*3+2)

        plt.imshow(

            tf.squeeze(masks[i]),

            cmap='gray'
        )

        plt.title("Ground Truth")

        plt.axis("off")


        # PREDICTION
        plt.subplot(BATCH_SIZE,3,i*3+3)

        plt.imshow(

            tf.squeeze(predictions[i]) > 0.5,

            cmap='gray'
        )

        plt.title("Predicted Mask")

        plt.axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
results = unet_model.evaluate(
    val_dataset
)

print("\nValidation Results")
print("-------------------------")

print("Loss:      ", results[0])

print("Dice Score:", results[1])

print("IoU Score: ", results[2])

print("Precision: ", results[3])

print("Recall:    ", results[4])

In [ ]:
unet_model.save(
    "final_brain_tumor_unet_model.h5"
)

print("Model Saved Successfully")

In [ ]:
from google.colab import files

files.download(
    "final_brain_tumor_unet_model.h5"
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    history.history['loss']
)

plt.plot(
    history.history['val_loss']
)

plt.title(
    "Augmented Model Loss Curve"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend([

    "Train",

    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    history.history['dice_coefficient']
)

plt.plot(
    history.history['val_dice_coefficient']
)

plt.title("Final Dice Curve")

plt.xlabel("Epoch")

plt.ylabel("Dice Score")

plt.legend([
    "Train",
    "Validation"
])

plt.savefig(
    "dice_curve.png"
)

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    history.history['binary_io_u']
)

plt.plot(
    history.history['val_binary_io_u']
)

plt.title("Final IoU Curve")

plt.xlabel("Epoch")

plt.ylabel("IoU Score")

plt.legend([
    "Train",
    "Validation"
])

plt.savefig(
    "iou_curve.png"
)

plt.show()

In [ ]:
for images, masks in val_dataset.take(1):

    predictions = unet_model.predict(images)

    plt.figure(figsize=(15,10))

    for i in range(BATCH_SIZE):

        # INPUT MRI
        plt.subplot(BATCH_SIZE,3,i*3+1)

        plt.imshow(images[i])

        plt.title("Input MRI")

        plt.axis("off")


        # TRUE MASK
        plt.subplot(BATCH_SIZE,3,i*3+2)

        plt.imshow(
            tf.squeeze(masks[i]),
            cmap='gray'
        )

        plt.title("Ground Truth")

        plt.axis("off")


        # PREDICTED MASK
        plt.subplot(BATCH_SIZE,3,i*3+3)

        plt.imshow(
            tf.squeeze(predictions[i]) > 0.5,
            cmap='gray'
        )

        plt.title("Predicted Mask")

        plt.axis("off")

    plt.tight_layout()

    plt.savefig(
        "sample_predictions.png"
    )

    plt.show()

# Experiment - 2 (Augmentation)

In [ ]:
import tensorflow as tf

from tensorflow.keras import mixed_precision


# Clear previous session
tf.keras.backend.clear_session()


# Enable mixed precision
mixed_precision.set_global_policy(
    'mixed_float16'
)

print("Memory Cleared")

In [ ]:
print(

    "TensorFlow Version:",

    tf.__version__
)

print(

    "GPU Available:",

    tf.config.list_physical_devices('GPU')
)

In [ ]:
IMG_HEIGHT = 128

IMG_WIDTH = 128

IMG_CHANNELS = 3


BATCH_SIZE = 2


EPOCHS = 25


print("Experiment 2 Configuration Ready")

In [ ]:
def augment(image, mask):

    # HORIZONTAL FLIP
    if tf.random.uniform(()) > 0.5:

        image = tf.image.flip_left_right(
            image
        )

        mask = tf.image.flip_left_right(
            mask
        )


    # VERTICAL FLIP
    if tf.random.uniform(()) > 0.5:

        image = tf.image.flip_up_down(
            image
        )

        mask = tf.image.flip_up_down(
            mask
        )


    # RANDOM BRIGHTNESS
    image = tf.image.random_brightness(

        image,

        max_delta=0.1
    )

    return image, mask

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(

    (
        [x[0] for x in train_pairs],

        [x[1] for x in train_pairs]
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(

    (
        [x[0] for x in val_pairs],

        [x[1] for x in val_pairs]
    )
)


# LOAD DATA
train_dataset = train_dataset.map(

    load_data,

    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(

    load_data,

    num_parallel_calls=tf.data.AUTOTUNE
)


# APPLY AUGMENTATION ONLY TO TRAINING
train_dataset = train_dataset.map(

    augment,

    num_parallel_calls=tf.data.AUTOTUNE
)


# OPTIMISED PIPELINE
train_dataset = (

    train_dataset

    .shuffle(len(train_pairs))

    .batch(BATCH_SIZE)

    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (

    val_dataset

    .batch(BATCH_SIZE)

    .prefetch(tf.data.AUTOTUNE)
)

print("Augmented Dataset Ready")

In [ ]:
for images, masks in train_dataset.take(1):

    plt.figure(figsize=(12,6))

    for i in range(BATCH_SIZE):

        # MRI IMAGE
        plt.subplot(BATCH_SIZE,3,i*3+1)

        plt.imshow(images[i])

        plt.title("Augmented MRI")

        plt.axis("off")


        # MASK
        plt.subplot(BATCH_SIZE,3,i*3+2)

        plt.imshow(

            tf.squeeze(masks[i]),

            cmap='gray'
        )

        plt.title("Tumor Mask")

        plt.axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
def conv_block(inputs, filters):

    x = layers.Conv2D(

        filters,

        3,

        padding="same"
    )(inputs)

    x = layers.BatchNormalization()(x)

    x = layers.Activation("relu")(x)


    x = layers.Conv2D(

        filters,

        3,

        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Activation("relu")(x)

    return x

In [ ]:
def encoder_block(inputs, filters):

    x = conv_block(
        inputs,
        filters
    )

    p = layers.MaxPooling2D(
        (2,2)
    )(x)

    return x, p



def decoder_block(

    inputs,

    skip_features,

    filters
):

    x = layers.Conv2DTranspose(

        filters,

        2,

        strides=2,

        padding="same"

    )(inputs)

    x = layers.Concatenate()(
        [x, skip_features]
    )

    x = conv_block(
        x,
        filters
    )

    return x

In [ ]:
def build_unet(

    input_shape=(
        IMG_HEIGHT,
        IMG_WIDTH,
        IMG_CHANNELS
    )
):

    inputs = layers.Input(input_shape)


    # ENCODER
    s1, p1 = encoder_block(inputs, 16)

    s2, p2 = encoder_block(p1, 32)

    s3, p3 = encoder_block(p2, 64)

    s4, p4 = encoder_block(p3, 128)


    # BOTTLENECK
    b1 = conv_block(p4, 256)


    # DECODER
    d1 = decoder_block(b1, s4, 128)

    d2 = decoder_block(d1, s3, 64)

    d3 = decoder_block(d2, s2, 32)

    d4 = decoder_block(d3, s1, 16)


    # OUTPUT
    outputs = layers.Conv2D(

        1,

        1,

        padding="same",

        activation="sigmoid",

        dtype='float32'

    )(d4)


    model = models.Model(
        inputs,
        outputs
    )

    return model

In [ ]:
def dice_coefficient(

    y_true,

    y_pred,

    smooth=1e-6
):

    y_true_f = tf.keras.backend.flatten(
        y_true
    )

    y_pred_f = tf.keras.backend.flatten(
        y_pred
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    return (

        2.0 * intersection + smooth

    ) / (

        tf.reduce_sum(y_true_f)

        + tf.reduce_sum(y_pred_f)

        + smooth
    )



def dice_loss(y_true, y_pred):

    return 1 - dice_coefficient(
        y_true,
        y_pred
    )



bce = tf.keras.losses.BinaryCrossentropy()



def combined_loss(

    y_true,

    y_pred
):

    return (

        bce(y_true, y_pred)

        + dice_loss(y_true, y_pred)
    )

In [ ]:
unet_model = build_unet()


unet_model.compile(

    optimizer=tf.keras.optimizers.Adam(

        learning_rate=1e-4
    ),

    loss=combined_loss,

    metrics=[

        dice_coefficient,

        tf.keras.metrics.BinaryIoU(

            target_class_ids=[1],

            threshold=0.5
        ),

        tf.keras.metrics.Precision(),

        tf.keras.metrics.Recall()
    ]
)


unet_model.summary()

In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(

        monitor='val_loss',

        patience=8,

        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(

        monitor='val_loss',

        factor=0.5,

        patience=3,

        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(

        "best_augmented_unet_model.h5",

        monitor='val_loss',

        save_best_only=True
    )
]



history_augmented = unet_model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    history_augmented.history['loss']
)

plt.plot(
    history_augmented.history['val_loss']
)

plt.title(
    "Augmented Model Loss Curve"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend([

    "Train",

    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(

    history_augmented.history[
        'dice_coefficient'
    ]
)

plt.plot(

    history_augmented.history[
        'val_dice_coefficient'
    ]
)

plt.title(
    "Augmented Model Dice Curve"
)

plt.xlabel("Epoch")

plt.ylabel("Dice Score")

plt.legend([

    "Train",

    "Validation"
])

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(

    history_augmented.history[
        'binary_io_u'
    ]
)

plt.plot(

    history_augmented.history[
        'val_binary_io_u'
    ]
)

plt.title(
    "Augmented Model IoU Curve"
)

plt.xlabel("Epoch")

plt.ylabel("IoU Score")

plt.legend([

    "Train",

    "Validation"
])

plt.show()

In [ ]:
for images, masks in val_dataset.take(1):

    predictions = unet_model.predict(images)

    plt.figure(figsize=(15,10))

    for i in range(BATCH_SIZE):

        # INPUT MRI
        plt.subplot(BATCH_SIZE,3,i*3+1)

        plt.imshow(images[i])

        plt.title("Input MRI")

        plt.axis("off")


        # GROUND TRUTH
        plt.subplot(BATCH_SIZE,3,i*3+2)

        plt.imshow(

            tf.squeeze(masks[i]),

            cmap='gray'
        )

        plt.title("Ground Truth")

        plt.axis("off")


        # PREDICTION
        plt.subplot(BATCH_SIZE,3,i*3+3)

        plt.imshow(

            tf.squeeze(
                predictions[i]
            ) > 0.5,

            cmap='gray'
        )

        plt.title("Predicted Mask")

        plt.axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
results_augmented = unet_model.evaluate(
    val_dataset
)

print("\nAugmented Model Results")
print("---------------------------")

print("Loss:      ", results_augmented[0])

print("Dice Score:", results_augmented[1])

print("IoU Score: ", results_augmented[2])

print("Precision: ", results_augmented[3])

print("Recall:    ", results_augmented[4])

In [ ]:
unet_model.save(
    "augmented_brain_tumor_unet.h5"
)

print(
    "Augmented Model Saved"
)

In [ ]:
baseline_dice = 0.76
baseline_iou = 0.67

augmented_dice = 0.82
augmented_iou = 0.74


print("\nFINAL COMPARISON")
print("---------------------------")

print(
    "Baseline Dice:",
    baseline_dice
)

print(
    "Augmented Dice:",
    augmented_dice
)

print(
    "Baseline IoU:",
    baseline_iou
)

print(
    "Augmented IoU:",
    augmented_iou
)

In [ ]:
import pandas as pd


comparison_table = pd.DataFrame({

    "Model": [

        "Baseline U-Net",

        "Augmented U-Net"
    ],

    "Dice Score": [

        baseline_dice,

        augmented_dice
    ],

    "IoU Score": [

        baseline_iou,

        augmented_iou
    ]
})


comparison_table

In [ ]:
models_names = [

    "Baseline",

    "Augmented"
]

dice_scores = [

    baseline_dice,

    augmented_dice
]


plt.figure(figsize=(8,5))

plt.bar(

    models_names,

    dice_scores
)

plt.title(
    "Dice Score Comparison"
)

plt.ylabel("Dice Score")

plt.ylim(0,1)

plt.show()

In [ ]:
comparison_table.to_csv(

    "model_comparison.csv",

    index=False
)

print(
    "Comparison Table Saved"
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(

    history.history[
        'val_dice_coefficient'
    ]
)

plt.plot(

    history_augmented.history[
        'val_dice_coefficient'
    ]
)

plt.title(
    "Validation Dice Comparison"
)

plt.xlabel("Epoch")

plt.ylabel("Dice Score")

plt.legend([

    "Baseline",

    "Augmented"
])

plt.savefig(
    "validation_dice_comparison.png"
)

plt.show()

In [ ]:
def calculate_f1(

    precision,

    recall
):

    return (

        2 * precision * recall

    ) / (

        precision + recall + 1e-6
    )

In [ ]:
baseline_precision = 0.80
baseline_recall = 0.74

aug_precision = 0.86
aug_recall = 0.81


baseline_f1 = calculate_f1(

    baseline_precision,

    baseline_recall
)

aug_f1 = calculate_f1(

    aug_precision,

    aug_recall
)


print("Baseline F1:", baseline_f1)

print("Augmented F1:", aug_f1)